In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [7]:
# ================================================================
# STAGE 1 — HUMAN BANGLA NEWS PASSAGES
# Fixed / faster / checkpointed version
#
# IMPORTANT:
# Keep your existing 100-topic TOPICS = [...] block unchanged.
# Paste this code AFTER your TOPICS block.
# ================================================================

import os
import sys
import re
import time
import pickle
import random
import hashlib
import unicodedata
import itertools
import statistics
from collections import Counter, defaultdict

# ------------------------------------------------
# INSTALL / IMPORT
# ------------------------------------------------
try:
    from datasets import load_dataset
except ImportError:
    os.system(f"{sys.executable} -m pip install -q datasets")
    from datasets import load_dataset

import pandas as pd


# ================================================================
# CONFIG
# ================================================================

SEED = 42
random.seed(SEED)

OUT_DIR = "/kaggle/working" if os.path.isdir("/kaggle/working") else "."

# Start with ONLY the main corpus.
# If coverage is poor, add XLSum later.
CORPORA = [
    dict(
        name="thedeba/bnnews",
        config=None,
        revisions=[None]
    )
]

MAX_DOCS_PER_CORPUS = 300_000

# Maximum runtime for one execution.
# Re-run the cell to continue from checkpoint.
TIME_LIMIT_MIN = 30

# Save checkpoint every N documents.
CKPT_EVERY = 25_000

# Number of candidates retained per topic.
K = 10

# Number written to human_candidates.csv.
N_ALTS = 8

# Passage length.
MIN_W = 100
MAX_W = 200
TARGET_W = 150

# Article length.
MIN_ARTICLE_WORDS = 130
MAX_ARTICLE_WORDS = 6000

# Minimum number of anchor/concept hits.
MIN_CORE_HITS = 2

# Candidate acceptance threshold.
ACCEPT = 50

# Stop scanning once this many topics have good candidates.
COVERAGE_TARGET = 95

# Small preference for primary corpus.
PRIMARY_BONUS = 4

# Do NOT save full article text.
SAVE_FULL_TEXT = False

CKPT = os.path.join(
    OUT_DIR,
    "stage1_checkpoint.pkl"
)

# ------------------------------- TOPICS -------------------------------
# (id, English topic, [required concept groups: first = anchor], support terms)
# Terms are word-PREFIXES (handles Bangla inflection). "x y" = adjacent words. "x$" = exact word.
AI  = "কৃত্রিম বুদ্ধিমত্তা|এআই|এ$ আই$|চ্যাটজিপিটি|জেনারেটিভ এআই|ai$|chatgpt$|মেশিন লার্নিং"
EDU = "শিক্ষা|শিক্ষার্থী|শিক্ষক|স্কুল|বিদ্যালয়|ক্লাসরুম|বিশ্ববিদ্যালয়|পড়াশোনা|পড়ালেখা"
HEA = "স্বাস্থ্য|চিকিৎসা|চিকিৎসক|রোগ|রোগী|হাসপাতাল|ডাক্তার|রোগ নির্ণয়"
YTH = "তরুণ|যুব|কিশোর|শিক্ষার্থী|নতুন প্রজন্ম|প্রজন্ম"
TOPICS = [
(1,"Traffic congestion in Dhaka",["যানজট|জ্যাম$|ট্রাফিক জ্যাম","ঢাকা"],"ট্রাফিক|সড়ক|যানবাহন|সিগন্যাল|রাজধানী|বাস|রিকশা|মেট্রোরেল|ফ্লাইওভার|ঘণ্টা"),
(2,"Air pollution in Bangladesh",["বায়ুদূষণ|বায়ু দূষণ|বায়ুমান|একিউআই|দূষিত বাতাস|দূষিত বায়ু|বাতাসের মান|বাতাস দূষণ"],"ধুলা|ধূলিকণা|ধোঁয়া|ইটভাটা|দূষণ|স্বাস্থ্য|বাতাস|নির্মাণকাজ|যানবাহন|ঢাকা"),
(3,"Water pollution in Bangladesh",["পানিদূষণ|পানি দূষণ|নদী দূষণ|নদীদূষণ|দূষিত পানি|পানির দূষণ|শিল্পবর্জ্য|তরল বর্জ্য|বুড়িগঙ্গা|বিষাক্ত পানি"],"নদী|বর্জ্য|কারখানা|ট্যানারি|খাল|দূষণ|পানি|রাসায়নিক|আর্সেনিক|পয়ঃনিষ্কাশন"),
(4,"Plastic waste management",["প্লাস্টিক","বর্জ্য|ময়লা|আবর্জনা|দূষণ|পলিথিন|রিসাইকেল|ব্যবস্থাপনা"],"ডাস্টবিন|সিটি করপোরেশন|পরিবেশ|নিষিদ্ধ|ব্যাগ|বোতল|সংগ্রহ"),
(5,"Flooding in Bangladesh",["বন্যা|প্লাবিত|পানিবন্দি|জলাবদ্ধতা"],"ত্রাণ|বৃষ্টি|ক্ষতিগ্রস্ত|বাঁধ|নদী|পানি|উজান|আশ্রয়কেন্দ্র|ঘরবাড়ি|ফসল"),
(6,"River erosion in Bangladesh",["ভাঙন|নদীগর্ভে|বিলীন","নদী|পদ্মা|যমুনা|মেঘনা|ব্রহ্মপুত্র|তিস্তা"],"বসতভিটা|ঘরবাড়ি|ফসলি জমি|বাঁধ|পানি উন্নয়ন বোর্ড|পাউবো|ক্ষতিগ্রস্ত|ভিটেমাটি|ভাঙনকবলিত|বর্ষা"),
(7,"Coastal erosion",["উপকূলীয় ভাঙন|উপকূল ভাঙন|সমুদ্রের ভাঙন|তীর ভাঙন|সৈকত ক্ষয়|উপকূল ক্ষয়|ভাঙন|বিলীন","উপকূল|সমুদ্র|সাগর|বঙ্গোপসাগর|কুতুবদিয়া|সন্দ্বীপ|হাতিয়া|কুয়াকাটা|মহেশখালী|সৈকত"],"বেড়িবাঁধ|জোয়ার|ঘূর্ণিঝড়|জলোচ্ছ্বাস|লোনা পানি|লবণাক্ত|বসতবাড়ি|বাঁধ"),
(8,"Rising sea levels",["সমুদ্রপৃষ্ঠ|সমুদ্র পৃষ্ঠ|সমুদ্রসমতল|সমুদ্র সমতল|সমুদ্রের উচ্চতা|সমুদ্রের পানির উচ্চতা|সমুদ্রের পানির স্তর|সাগরের পানির স্তর"],"জলবায়ু|উপকূল|লবণাক্ত|জলোচ্ছ্বাস|বৃদ্ধি|গলে|হিমবাহ|বরফ|তলিয়ে|ডুবে"),
(9,"Cyclone preparedness",["ঘূর্ণিঝড়","প্রস্তুতি|আশ্রয়কেন্দ্র|সতর্ক|সংকেত|স্বেচ্ছাসেবক|সিপিপি|মহড়া|প্রস্তুত"],"উপকূল|উপকূলীয়|আবহাওয়া|জেলা|নৌযান|ট্রলার|মাছ|ত্রাণ"),
(10,"Urbanization in Bangladesh",["নগরায়ণ|নগরায়ন|শহরায়ণ|শহরায়ন|অপরিকল্পিত নগর|শহরমুখী|নগর পরিকল্পনা|নগরের বিস্তার"],"শহর|নগর|বস্তি|জনসংখ্যা|ঢাকা|অবকাঠামো|পরিকল্পনা|আবাসন|ভবন|গ্রাম থেকে"),
(11,"Rural development",["গ্রামীণ উন্নয়ন|গ্রামীণ অর্থনীতি|পল্লী উন্নয়ন|পল্লি উন্নয়ন|গ্রামের উন্নয়ন|গ্রামাঞ্চলের উন্নয়ন|গ্রামীণ জনপদ|আমার গ্রাম আমার শহর"],"গ্রাম|গ্রামীণ|পল্লী|পল্লি|কৃষি|সড়ক|বিদ্যুৎ|এলজিইডি|ইউনিয়ন|উপজেলা|দারিদ্র্য"),
(12,"Population growth",["জনসংখ্যা","বৃদ্ধি|বিস্ফোরণ|নিয়ন্ত্রণ|পরিবার পরিকল্পনা|জন্মহার|আদমশুমারি|শুমারি|ঘনত্ব"],"জন্ম|প্রজনন|জনশুমারি|বিবিএস|জনমিতিক|ঘনবসতি"),
(13,"Youth unemployment",["বেকারত্ব|বেকার|কর্মসংস্থান","তরুণ|যুব|যুবক|শিক্ষিত|স্নাতক"],"চাকরি|দক্ষতা|শ্রমবাজার|বিবিএস|শিক্ষিত|শতাংশ|সনদ"),
(14,"Women in the workforce",["নারী","কর্মসংস্থান|কর্মক্ষেত্র|শ্রমশক্তি|শ্রমবাজার|পোশাকশ্রমিক|চাকরি|কর্মজীবী|উদ্যোক্তা|শ্রমিক"],"পোশাক|গার্মেন্টস|বেতন|মজুরি|বৈষম্য|কর্মস্থল|অর্থনীতি|অংশগ্রহণ"),
(15,"Child education",["শিশু","শিক্ষা|স্কুল|বিদ্যালয়|প্রাথমিক|পড়ালেখা|পাঠদান"],"অভিভাবক|শিক্ষক|ভর্তি|উপবৃত্তি|ঝরে|পাঠ্যবই|শ্রেণি"),
(16,"Rural literacy",["সাক্ষরতা|নিরক্ষর|অক্ষরজ্ঞান|স্বাক্ষরতা","গ্রাম|গ্রামীণ|গ্রামাঞ্চল|পল্লী"],"শিক্ষা|হার|বয়স্ক|বিবিএস|উপানুষ্ঠানিক|নারী"),
(17,"Public transportation",["গণপরিবহন|গণপরিবহণ|গণ পরিবহন|পাবলিক বাস|পাবলিক ট্রান্সপোর্ট|মেট্রোরেল|মেট্রো রেল|বিআরটিসি|লোকাল বাস|বাস রুট|বাস সার্ভিস|সিটি সার্ভিস|বিআরটি"],"যাত্রী|ভাড়া|বাস|ট্রেন|রেল|সড়ক|ঢাকা|পরিবহন|যানজট|রুট|ভিড়"),
(18,"Road safety",["সড়ক দুর্ঘটনা|সড়ক নিরাপত্তা|নিরাপদ সড়ক|সড়কে প্রাণ|সড়কে মৃত্যু|সড়ক পরিবহন আইন|হেলমেট|ট্রাফিক আইন|সড়কে দুর্ঘটনা"],"চালক|লাইসেন্স|গতিসীমা|বেপরোয়া|ফিটনেস|মোটরসাইকেল|নিহত|আহত|বাস|যাত্রী"),
(19,"Food safety",["নিরাপদ খাদ্য|ভেজাল|ফরমালিন|খাদ্যে ভেজাল|খাদ্যমান|মেয়াদোত্তীর্ণ|অস্বাস্থ্যকর|খাদ্য নিরাপত্তা আইন|বিএসটিআই"],"খাদ্য|ভোক্তা|রেস্তোরাঁ|অভিযান|জরিমানা|পরীক্ষা|কার্বাইড|রাসায়নিক|ক্ষতিকর"),
(20,"Rising prices of essential goods",["নিত্যপণ্য|নিত্যপ্রয়োজনীয়|দ্রব্যমূল্য|পণ্যের দাম|চালের দাম|ডালের দাম|তেলের দাম|চিনির দাম|সবজির দাম|বাজারদর|বাজারে দাম"],"বেড়েছে|বেড়ে|বাড়তি|বাড়ছে|ঊর্ধ্বগতি|ঊর্ধ্বমুখী|চড়া|অস্থির|বাজার|ক্রেতা|সিন্ডিকেট|টিসিবি|কেজি|পেঁয়াজ"),
(21,"AI in education",[AI,EDU],"প্রযুক্তি|শিখন|পাঠদান|অ্যাসাইনমেন্ট|গবেষণা|শিক্ষাব্যবস্থা|টুল"),
(22,"AI in healthcare",[AI,HEA],"রোগ নির্ণয়|ওষুধ|প্রযুক্তি|ক্যানসার|এক্স রে|টেলিমেডিসিন|ডেটা|চিকিৎসাসেবা"),
(23,"AI and employment",[AI,"চাকরি|কর্মসংস্থান|কর্মী|পেশা|শ্রমবাজার|কর্মক্ষেত্র|বেকার|ছাঁটাই|নিয়োগ|কর্মজীবী|পেশাজীবী"],"অটোমেশন|দক্ষতা|ভবিষ্যৎ|প্রযুক্তি|প্রতিস্থাপন|ঝুঁকি"),
(24,"Social media algorithms",["অ্যালগরিদম|এলগরিদম|অ্যালগোরিদম","সামাজিক যোগাযোগ|সামাজিক মাধ্যম|ফেসবুক|ইউটিউব|টিকটক|ইনস্টাগ্রাম|সোশ্যাল মিডিয়া"],"ফিড|কনটেন্ট|ব্যবহারকারী|বিজ্ঞাপন|আসক্তি|মেটা|গুগল"),
(25,"Online privacy",["গোপনীয়তা|ব্যক্তিগত তথ্য|প্রাইভেসি|তথ্য ফাঁস|তথ্য চুরি","অনলাইন|ইন্টারনেট|সাইবার|ডিজিটাল|ফেসবুক|সামাজিক যোগাযোগ|অ্যাপ|ওয়েবসাইট"],"ডেটা|সুরক্ষা|পাসওয়ার্ড|হ্যাক|কুকি|ব্যবহারকারী|ট্র্যাক"),
(26,"Cybersecurity",["সাইবার|হ্যাকার|হ্যাকিং|হ্যাক"],"তথ্য|নিরাপত্তা|ডেটা|পাসওয়ার্ড|ম্যালওয়্যার|র‍্যানসমওয়্যার|ফিশিং|ফায়ারওয়াল|আইন|হামলা"),
(27,"Smartphone addiction",["আসক্তি|আসক্ত|নেশা","স্মার্টফোন|মোবাইল|ফোন|মুঠোফোন|গেম|ইন্টারনেট|সামাজিক যোগাযোগ|ফেসবুক|ডিজিটাল ডিভাইস|স্ক্রিন"],"শিশু|কিশোর|ঘুম|চোখ|মানসিক|সময়|অভিভাবক"),
(28,"Digital payments",["ডিজিটাল পেমেন্ট|ডিজিটাল লেনদেন|ডিজিটাল অর্থ|অনলাইন পেমেন্ট|অনলাইনে লেনদেন|অনলাইন লেনদেন|ক্যাশলেস|ডিজিটাল ওয়ালেট|কিউআর কোড|পেমেন্ট গেটওয়ে|কার্ডে লেনদেন|ডিজিটাল আর্থিক|ইলেকট্রনিক লেনদেন"],"লেনদেন|কার্ড|মোবাইল ব্যাংকিং|বিকাশ|নগদ|রকেট|ব্যাংক|পেমেন্ট|অ্যাপ|বাংলা কিউআর"),
(29,"Mobile banking",["মোবাইল ব্যাংকিং|মোবাইল আর্থিক|এমএফএস|বিকাশ|মোবাইল ব্যাংক|মোবাইলে টাকা"],"এজেন্ট|লেনদেন|ক্যাশ ইন|ক্যাশ আউট|রকেট|নগদ|গ্রাহক|ব্যাংক|রেমিট্যান্স|অ্যাকাউন্ট"),
(30,"E-commerce growth",["ই কমার্স|ইকমার্স|এফ কমার্স|অনলাইন শপিং|অনলাইন ব্যবসা|অনলাইন কেনাকাটা|ডিজিটাল কমার্স"],"বৃদ্ধি|বেড়ে|প্রসার|প্রবৃদ্ধি|দারাজ|অর্ডার|ডেলিভারি|বিক্রি|গ্রাহক|প্ল্যাটফর্ম|উদ্যোক্তা|ই ক্যাব"),
(31,"Self-driving cars",["চালকবিহীন|স্বচালিত|স্বয়ংচালিত|সেলফ ড্রাইভিং|চালকহীন|স্বয়ংক্রিয় গাড়ি|চালক ছাড়া|ড্রাইভারবিহীন|অটোনোমাস"],"গাড়ি|টেসলা|ওয়েমো|সেন্সর|প্রযুক্তি|ক্যামেরা|সড়ক|যানবাহন|কৃত্রিম বুদ্ধিমত্তা"),
(32,"Robots in the workplace",["রোবট|রোবটিক|রোবোটিক","কারখানা|কর্মী|কর্মক্ষেত্র|শ্রমিক|চাকরি|অফিস|শিল্প|গুদাম|উৎপাদন|কর্মসংস্থান"],"অটোমেশন|প্রযুক্তি|দক্ষতা|কৃত্রিম বুদ্ধিমত্তা|মানুষের"),
(33,"Cloud computing",["ক্লাউড"],"ডেটা|সার্ভার|সফটওয়্যার|ডেটা সেন্টার|অ্যামাজন|মাইক্রোসফট|গুগল|প্রযুক্তি|সংরক্ষণ|আইটি"),
(34,"Rural internet access",["ইন্টারনেট|ব্রডব্যান্ড|ওয়াইফাই|নেটওয়ার্ক","গ্রাম|গ্রামীণ|গ্রামাঞ্চল|পল্লী|প্রত্যন্ত|দুর্গম|ইউনিয়ন"],"সংযোগ|গতি|ডিজিটাল|টেলিকম|মোবাইল|ফাইবার|অপটিক্যাল|সেবা|তথ্যসেবা"),
(35,"5G technology",["ফাইভ জি|ফাইভজি|5 জি|5জি|5g$"],"নেটওয়ার্ক|স্পেকট্রাম|টেলিকম|গতি|অপারেটর|গ্রামীণফোন|রবি|বাংলালিংক|ফোর জি|4 জি|প্রযুক্তি|নিলাম|বিটিআরসি"),
(36,"Facial recognition",["ফেস রিকগনিশন|মুখাবয়ব শনাক্ত|মুখ শনাক্ত|ফেসিয়াল রিকগনিশন|মুখমণ্ডল শনাক্ত|চেহারা শনাক্ত|ফেস আইডি|মুখ চেনা"],"ক্যামেরা|নজরদারি|গোপনীয়তা|বায়োমেট্রিক|প্রযুক্তি|কৃত্রিম বুদ্ধিমত্তা|পুলিশ|সিসিটিভি|শনাক্ত"),
(37,"Virtual reality",["ভার্চুয়াল রিয়েলিটি|ভার্চুয়াল বাস্তবতা|ভিআর|ভার্চ্যুয়াল রিয়েলিটি|অগমেন্টেড রিয়েলিটি|মেটাভার্স|ভার্চুয়াল জগৎ"],"হেডসেট|গেম|প্রযুক্তি|ত্রিমাত্রিক|অকুলাস|মেটা|অভিজ্ঞতা|সিমুলেশন|এআর"),
(38,"Internet of Things",["ইন্টারনেট অব থিংস|ইন্টারনেট অফ থিংস|আইওটি|স্মার্ট ডিভাইস|স্মার্ট হোম"],"ডিভাইস|সেন্সর|স্মার্ট|নেটওয়ার্ক|প্রযুক্তি|স্বয়ংক্রিয়|ইন্টারনেট|সংযোগ|ডেটা"),
(39,"Data privacy",["ডেটা সুরক্ষা|তথ্য সুরক্ষা|ডেটা প্রাইভেসি|তথ্য গোপনীয়তা|ডেটা গোপনীয়তা|ব্যক্তিগত ডেটা|ডেটা ফাঁস|ডাটা সুরক্ষা|ডাটা ফাঁস|তথ্যের সুরক্ষা|তথ্যের গোপনীয়তা|ব্যক্তিগত তথ্য সুরক্ষা"],"ব্যবহারকারী|আইন|নিরাপত্তা|তথ্য|সম্মতি|কোম্পানি|হ্যাক|অধ্যাদেশ"),
(40,"Future smartphones",["স্মার্টফোন|ফোল্ডেবল|আইফোন|স্যামসাং|হুয়াওয়ে|শাওমি","ভবিষ্য|আগামী|আসছে|আসতে|উন্মোচন|লঞ্চ|নতুন মডেল|আসবে|নতুন প্রযুক্তি"],"ফিচার|ক্যামেরা|ব্যাটারি|প্রসেসর|ডিসপ্লে|এআই|প্রযুক্তি|বাজারে"),
(41,"Online education",["অনলাইন ক্লাস|অনলাইনে ক্লাস|অনলাইন শিক্ষা|অনলাইনে পাঠদান|ই লার্নিং|অনলাইন পাঠদান|দূরশিক্ষণ|ডিজিটাল ক্লাস|অনলাইন লার্নিং|ভার্চুয়াল ক্লাস|জুম ক্লাস"],"শিক্ষার্থী|শিক্ষক|ইন্টারনেট|করোনা|স্কুল|জুম|পাঠদান|স্মার্টফোন|প্ল্যাটফর্ম"),
(42,"Online examinations",["অনলাইন পরীক্ষা|অনলাইনে পরীক্ষা|অনলাইন মূল্যায়ন|ডিজিটাল পরীক্ষা|অনলাইন ভর্তি পরীক্ষা|অনলাইন কুইজ"],"পরীক্ষা|শিক্ষার্থী|ইন্টারনেট|পরীক্ষার্থী|জুম|প্রশ্ন|মূল্যায়ন|নকল|ফল"),
(43,"University admissions",["ভর্তি পরীক্ষা|ভর্তি|গুচ্ছ ভর্তি","বিশ্ববিদ্যালয়|ভার্সিটি|ঢাবি|বুয়েট|জাবি|চবি|রাবি|গুচ্ছ"],"আসন|আবেদন|ফল|ইউনিট|জিপিএ|মেধাতালিকা|শিক্ষার্থী|ফি|ভর্তিচ্ছু"),
(44,"Standardized testing",["স্ট্যান্ডার্ডাইজড|মানসম্মত পরীক্ষা|প্রমিত পরীক্ষা|পাবলিক পরীক্ষা|এমসিকিউ|বহুনির্বাচনী|সৃজনশীল পদ্ধতি|পরীক্ষা পদ্ধতি|মূল্যায়ন পদ্ধতি|জিআরই|স্যাট$|আইইএলটিএস|টোফেল|জিমেট"],"পরীক্ষা|প্রশ্ন|নম্বর|মূল্যায়ন|শিক্ষার্থী|পরীক্ষার্থী|পাসের হার|জিপিএ|বোর্ড"),
(45,"Practical education",["ব্যবহারিক|প্রায়োগিক|হাতে কলমে|কর্মমুখী শিক্ষা|প্র্যাকটিক্যাল|বাস্তবমুখী|দক্ষতাভিত্তিক","শিক্ষা|শিক্ষার্থী|শিক্ষক|বিদ্যালয়|স্কুল|কলেজ|বিশ্ববিদ্যালয়|ক্লাস"],"দক্ষতা|পাঠ্যক্রম|শিক্ষাক্রম|ল্যাব|ইন্টার্নশিপ|শিল্প|বাস্তব|কর্মমুখী"),
(46,"Student academic pressure",["চাপ$|চাপে$|চাপের$|মানসিক চাপ|পড়ার চাপ|পরীক্ষার চাপ|পড়াশোনার চাপ|একাডেমিক চাপ|দুশ্চিন্তা|উৎকণ্ঠা","শিক্ষার্থী|ছাত্র|ছাত্রছাত্রী|পরীক্ষার্থী|কিশোর"],"পরীক্ষা|পড়াশোনা|পড়ালেখা|অভিভাবক|কোচিং|জিপিএ|মানসিক|প্রতিযোগিতা|ফল"),
(47,"School dropout",["ঝরে পড়া|ঝরে পড়ে|ঝরে যাচ্ছে|ঝরে পড়ছে|ড্রপআউট|ড্রপ আউট|স্কুল ছাড়|বিদ্যালয় ছাড়"],"শিক্ষার্থী|স্কুল|বিদ্যালয়|বাল্যবিয়ে|দারিদ্র্য|প্রাথমিক|মাধ্যমিক|শিশুশ্রম|হার|উপবৃত্তি"),
(48,"Learning technology",["শিক্ষা প্রযুক্তি|এডটেক|প্রযুক্তিনির্ভর শিক্ষা|ডিজিটাল শিক্ষা|ডিজিটাল ক্লাসরুম|স্মার্ট ক্লাসরুম|শিক্ষায় প্রযুক্তি|প্রযুক্তি নির্ভর শিক্ষা|শিখন প্রযুক্তি|মাল্টিমিডিয়া ক্লাসরুম|ডিজিটাল কনটেন্ট|শিক্ষায় আইসিটি|আইসিটি শিক্ষা|ডিজিটাল শিখন"],"শিক্ষার্থী|শিক্ষক|ক্লাসরুম|অ্যাপ|ডিজিটাল|পাঠ|ইন্টারনেট|কম্পিউটার|স্কুল|ল্যাপটপ"),
(49,"Private tutoring",["কোচিং|প্রাইভেট টিউশন|প্রাইভেট টিউটর|প্রাইভেট পড়া|গৃহশিক্ষক|টিউশনি|টিউশন"],"শিক্ষক|শিক্ষার্থী|অভিভাবক|নীতিমালা|বাণিজ্য|ফি|পরীক্ষা|পড়ালেখা|স্কুল"),
(50,"Extracurricular activities",["সহপাঠ্য|সহপাঠক্রমিক|সহপাঠ কার্যক্রম|সহশিক্ষা কার্যক্রম|সহশিক্ষাক্রমিক|পাঠ্যবহির্ভূত|পাঠ্যক্রম বহির্ভূত|বিতর্ক|স্কাউট|রোভার|বিএনসিসি|ক্লাব","শিক্ষার্থী|স্কুল|বিদ্যালয়|কলেজ|বিশ্ববিদ্যালয়"],"খেলাধুলা|সাংস্কৃতিক|প্রতিযোগিতা|দক্ষতা|নেতৃত্ব|ক্রীড়া|সংগঠন"),
(51,"Studying abroad",["বিদেশে পড়াশোনা|বিদেশে পড়তে|বিদেশে পড়ালেখা|বিদেশে উচ্চশিক্ষা|উচ্চশিক্ষার্থে|স্টুডেন্ট ভিসা|শিক্ষার্থী ভিসা|বিদেশি বিশ্ববিদ্যালয়|বিদেশে শিক্ষা|উচ্চশিক্ষা"],"ভিসা|শিক্ষার্থী|বিশ্ববিদ্যালয়|স্কলারশিপ|আইইএলটিএস|কানাডা|অস্ট্রেলিয়া|যুক্তরাষ্ট্র|যুক্তরাজ্য|মালয়েশিয়া|জার্মানি|খরচ"),
(52,"International scholarships",["বৃত্তি$|বৃত্তির$|স্কলারশিপ|ফুলব্রাইট|চেভনিং|এরাসমাস|ডিএএডি|কমনওয়েলথ","আন্তর্জাতিক|বিদেশ|বিদেশি|যুক্তরাষ্ট্র|যুক্তরাজ্য|জাপান|জার্মানি|অস্ট্রেলিয়া|কানাডা|চীন|কোরিয়া|মালয়েশিয়া|তুরস্ক|ইউরোপ|ভারত|বিশ্ববিদ্যালয়"],"আবেদন|শিক্ষার্থী|স্নাতকোত্তর|পিএইচডি|ভর্তি|সুযোগ|পূর্ণ অর্থায়ন"),
(53,"Vocational education",["কারিগরি|বৃত্তিমূলক|ভোকেশনাল|টেকনিক্যাল|পলিটেকনিক|টিভিইটি|দক্ষতা উন্নয়ন"],"শিক্ষা|প্রশিক্ষণ|দক্ষতা|ডিপ্লোমা|শিক্ষার্থী|কর্মসংস্থান|কর্মমুখী|ইনস্টিটিউট"),
(54,"English education",["ইংরেজি","শিক্ষা|শিক্ষার্থী|শিক্ষক|স্কুল|বিদ্যালয়|ক্লাস|পাঠ|পড়া"],"ভাষা|দক্ষতা|শেখা|পাঠ্যবই|পরীক্ষা|কোর্স|মাধ্যম|ব্যাকরণ"),
(55,"AI-assisted learning",[AI,"শিক্ষার্থী|শিখন|শেখা|শেখার|শেখাতে|শিক্ষা|পড়া|টিউটর|শিক্ষক|পাঠ"],"ব্যক্তিগত|শিক্ষক|টিউটর|প্রযুক্তি|অ্যাপ|শেখান|পাঠদান|শিক্ষাব্যবস্থা"),
(56,"Climate change",["জলবায়ু"],"তাপমাত্রা|কার্বন|নির্গমন|অভিযোজন|ক্ষতিগ্রস্ত|প্যারিস চুক্তি|তহবিল|বৈশ্বিক|উষ্ণায়ন|ঝুঁকি|দুর্যোগ|খরা"),
(57,"Global warming",["বৈশ্বিক উষ্ণায়ন|বিশ্ব উষ্ণায়ন|গ্লোবাল ওয়ার্মিং|উষ্ণায়ন|বৈশ্বিক উষ্ণতা|বিশ্ব উষ্ণতা|তাপমাত্রা বৃদ্ধি|তাপমাত্রা বেড়"],"কার্বন|গ্রিনহাউস|নির্গমন|বরফ|গলে|হিমবাহ|সমুদ্র|জলবায়ু|তাপপ্রবাহ|ডিগ্রি|পৃথিবী"),
(58,"Renewable energy",["নবায়নযোগ্য|নবায়ণযোগ্য|পুনর্নবীকরণযোগ্য|সবুজ জ্বালানি|রিনিউয়েবল|পরিচ্ছন্ন জ্বালানি"],"জ্বালানি|বিদ্যুৎ|সৌর|বায়ু|জীবাশ্ম|কার্বন|মেগাওয়াট|বিনিয়োগ|সোলার|বায়োগ্যাস|লক্ষ্যমাত্রা|শক্তি"),
(59,"Solar energy",["সৌর$|সৌরবিদ্যুৎ|সৌরশক্তি|সৌরচালিত|সৌর প্যানেল|সৌর বিদ্যুৎ|সোলার|রুফটপ"],"বিদ্যুৎ|প্যানেল|মেগাওয়াট|ইডকল|নবায়নযোগ্য|জ্বালানি|সেচ|সূর্যের আলো|ব্যাটারি"),
(60,"Wind energy",["বায়ুবিদ্যুৎ|বায়ু বিদ্যুৎ|বায়ুশক্তি|বায়ু শক্তি|বায়ুকল|উইন্ড|বায়ুচালিত|বায়ু টারবাইন|বাতাস থেকে বিদ্যুৎ|পবনশক্তি"],"টারবাইন|বিদ্যুৎ|মেগাওয়াট|উপকূল|জ্বালানি|নবায়নযোগ্য|পতেঙ্গা|কুতুবদিয়া|প্রকল্প|শক্তি|সমুদ্র"),
(61,"Electric vehicles",["ইলেকট্রিক গাড়ি|বৈদ্যুতিক গাড়ি|ইলেকট্রিক ভেহিকল|বৈদ্যুতিক যান|ইভি$|ইলেকট্রিক বাস|বৈদ্যুতিক বাস|ইলেকট্রিক কার|ইলেকট্রিক বাইক|বৈদ্যুতিক বাইক|বৈদ্যুতিক মোটরসাইকেল|ইলেকট্রিক মোটরসাইকেল|ইলেকট্রিক স্কুটার|চার্জিং স্টেশন|টেসলা|বিওয়াইডি"],"ব্যাটারি|চার্জ|জ্বালানি|পরিবেশবান্ধব|গাড়ি|শুল্ক|আমদানি|কার্বন|কিলোমিটার|পেট্রল"),
(62,"Deforestation",["উজাড়|বননিধন|বন নিধন|বৃক্ষনিধন|গাছ কাটা|গাছ কেটে|বনভূমি|বনাঞ্চল|বন দখল|বন ধ্বংস|বন কেটে|বন বিনাশ"],"বন|বনবিভাগ|সুন্দরবন|পাহাড়|গাছ|বৃক্ষ|দখল|পরিবেশ|বন্যপ্রাণী|কাঠ|বনায়ন|সংরক্ষিত"),
(63,"Wildlife conservation",["বন্যপ্রাণী|বন্য প্রাণী|বাঘ|হাতি|ডলফিন|কচ্ছপ|গন্ডার|রয়েল বেঙ্গল|বেঙ্গল টাইগার|বিপন্ন প্রজাতি|বিপন্ন প্রাণী|বিলুপ্তপ্রায়","সংরক্ষণ|রক্ষা|সুরক্ষা|সংরক্ষিত|অভয়ারণ্য|সংকটাপন্ন|বিপন্ন|বিলুপ্ত|শিকার|পাচার"],"অভয়ারণ্য|বন বিভাগ|বন অধিদপ্তর|জাতীয় উদ্যান|সুন্দরবন|প্রজাতি|চোরাশিকার|আবাসস্থল|হরিণ"),
(64,"Ocean pollution",["সাগর|সমুদ্র|মহাসাগর|বঙ্গোপসাগর|সামুদ্রিক","দূষণ|দূষিত|প্লাস্টিক|তেল ছড়ি|বর্জ্য|আবর্জনা|মাইক্রোপ্লাস্টিক"],"মাছ|প্রাণী|প্লাস্টিক|তেল|জাহাজ|জাহাজভাঙা|জীববৈচিত্র্য|উপকূল"),
(65,"Coral reefs",["প্রবাল|কোরাল"],"সেন্টমার্টিন|ছেঁড়াদ্বীপ|সমুদ্র|সাগর|দ্বীপ|জীববৈচিত্র্য|সামুদ্রিক|মাছ|পর্যটক|ব্লিচিং|সংরক্ষণ"),
(66,"Biodiversity loss",["জীববৈচিত্র্য|জীব বৈচিত্র্য|জীববৈচিত্র|বাস্তুতন্ত্র|প্রতিবেশ|প্রজাতি বিলুপ্ত|বিপন্ন প্রজাতি|বিলুপ্তির ঝুঁকি|বিলুপ্তির হুমকি"],"প্রজাতি|প্রাণী|উদ্ভিদ|আবাসস্থল|বন|পরিবেশ|হুমকি|সংরক্ষণ|ক্ষতি|হ্রাস"),
(67,"Sustainable agriculture",["টেকসই কৃষি|টেকসই চাষ|পরিবেশবান্ধব কৃষি|সহনশীল কৃষি|জলবায়ু সহনশীল|সমন্বিত বালাই|আইপিএম|ফসল আবর্তন|পরিবেশবান্ধব চাষ|টেকসই খাদ্য"],"কৃষক|কৃষি|ফসল|মাটি|সার|কীটনাশক|উৎপাদন|পানি|জলবায়ু|জৈব|খরা|লবণসহিষ্ণু|জাত"),
(68,"Organic farming",["জৈব কৃষি|জৈব চাষ|জৈব সার|অর্গানিক|জৈব পদ্ধতি|রাসায়নিক সারমুক্ত|বিষমুক্ত|রাসায়নিকমুক্ত|কীটনাশকমুক্ত|ভার্মিকম্পোস্ট|কেঁচো সার|জৈব সবজি|জৈব ফসল"],"কৃষক|ফসল|সার|কীটনাশক|রাসায়নিক|মাটি|সবজি|খামার|চাষ|কম্পোস্ট|ফলন|স্বাস্থ্য"),
(69,"Water conservation",["পানি সংরক্ষণ|পানি সাশ্রয়|পানির অপচয়|পানি অপচয়|বৃষ্টির পানি|ভূগর্ভস্থ পানি|পানির স্তর|পানি সংকট|পানির সংকট|পানির সাশ্রয়|জল সংরক্ষণ|পানি ব্যবস্থাপনা|পানির সংরক্ষণ"],"ভূগর্ভস্থ|ওয়াসা|সেচ|বৃষ্টি|সংকট|অপচয়|পুকুর|স্তর|সাশ্রয়|টেকসই|নদী"),
(70,"Recycling",["রিসাইকেল|পুনর্ব্যবহার|পুনঃব্যবহার|পুনঃপ্রক্রিয়া|পুনঃচক্রায়ন|রিসাইক্লিং|পুনর্চক্রায়ন|পুনর্ব্যবহৃত"],"বর্জ্য|প্লাস্টিক|কাগজ|কাচ|ধাতু|কারখানা|ভাঙারি|ময়লা|আবর্জনা|পরিবেশ|সংগ্রহ|বোতল|কম্পোস্ট"),
(71,"Physical exercise",["ব্যায়াম|শরীরচর্চা|শারীরিক পরিশ্রম|শারীরিক কার্যকলাপ|ওয়ার্কআউট|যোগব্যায়াম|কার্ডিও|শরীর চর্চা|ফিটনেস"],"স্বাস্থ্য|হৃদ|ওজন|পেশি|শরীর|নিয়মিত|মিনিট|সুস্থ|জিম|দৌড়|হাঁটা|রোগ"),
(72,"Healthy eating",["স্বাস্থ্যকর খাবার|স্বাস্থ্যকর খাদ্য|সুষম খাদ্য|সুষম খাবার|পুষ্টিকর খাবার|পুষ্টিকর খাদ্য|ডায়েট|খাদ্যাভ্যাস|খাদ্য তালিকা|পুষ্টিবিদ|স্বাস্থ্যসম্মত খাবার"],"ফল|সবজি|শাক|প্রোটিন|ভিটামিন|ক্যালরি|শর্করা|চর্বি|ওজন|চিনি|লবণ|পুষ্টি|পানি"),
(73,"Sleep and productivity",["ঘুম|নিদ্রা|অনিদ্রা","কর্মক্ষমতা|উৎপাদনশীলতা|মনোযোগ|দক্ষতা|কার্যক্ষমতা|একাগ্রতা|পারফরম্যান্স|কর্মদক্ষতা|সৃজনশীলতা|স্মৃতিশক্তি"],"ঘণ্টা|ক্লান্তি|ক্লান্ত|স্বাস্থ্য|সুস্থ|মস্তিষ্ক|রাত|অফিস|পড়াশোনা|মানসিক"),
(74,"Obesity among young people",["স্থূলতা|স্থূল|ওবেসিটি|অতিরিক্ত ওজন|বাড়তি ওজন|ওজনাধিক্য|বিএমআই","তরুণ|কিশোর|শিশু|শিক্ষার্থী|যুব|কিশোরী|ছাত্র|স্কুলের"],"ডায়াবেটিস|ফাস্ট ফুড|ওজন|খাদ্যাভ্যাস|ব্যায়াম|হৃদরোগ|চর্বি|স্বাস্থ্য|জাঙ্ক ফুড|কোমল পানীয়|শারীরিক পরিশ্রম|স্ক্রিন"),
(75,"Fast food",["ফাস্ট ফুড|ফাস্টফুড|জাঙ্ক ফুড|জাংক ফুড|বার্গার|পিৎজা|পিজা|ফ্রাইড চিকেন|প্রক্রিয়াজাত খাবার|কেএফসি"],"স্বাস্থ্য|চর্বি|ক্যালরি|চিপস|কোমল পানীয়|তেল|ওজন|রেস্তোরাঁ|খাবার|শিশু|তরুণ|ডেলিভারি|খাদ্যাভ্যাস"),
(76,"Mental health awareness",["মানসিক স্বাস্থ্য|মানসিক রোগ|মনোরোগ|মনোবিদ|মনোচিকিৎসক|বিষণ্ন|ডিপ্রেশন|মানসিক সুস্থতা|মানসিক সমস্যা"],"সচেতনতা|সচেতন|কাউন্সেলিং|থেরাপি|দুশ্চিন্তা|কলঙ্ক|সামাজিক|সহায়তা|ডব্লিউএইচও|দিবস|সেবা|উদ্বেগ"),
(77,"Smoking among young people",["ধূমপান|ধূমপায়ী|তামাক|সিগারেট|ভেপ$|ভ্যাপ$|ই সিগারেট|নিকোটিন|তামাকজাত|বিড়ি","তরুণ|কিশোর|যুব|শিক্ষার্থী|ছাত্র|স্কুল|কলেজ|বিশ্ববিদ্যালয়|শিশু|প্রজন্ম"],"স্বাস্থ্য|ক্যানসার|আসক্তি|নেশা|আইন|বিজ্ঞাপন|কর|দোকান"),
(78,"Screen time",["স্ক্রিন টাইম|স্ক্রিনে|স্ক্রিন|স্ক্রীন|ডিজিটাল ডিভাইস|মোবাইল ব্যবহার|মোবাইলের ব্যবহার|ফোন ব্যবহার|স্মার্টফোন ব্যবহার|টিভি দেখা|গেম খেলা|গেমিং","শিশু|কিশোর|শিক্ষার্থী|তরুণ|সন্তান|অভিভাবক"],"ঘণ্টা|সময়|অতিরিক্ত|আসক্তি|সীমিত|চোখ|ঘুম|দৃষ্টি"),
(79,"Benefits of walking",["হাঁটা|হাঁটলে|হাঁটার|হাঁটতে|হেঁটে|হাঁটুন|প্রাতঃভ্রমণ|ওয়াকিং","স্বাস্থ্য|উপকার|সুফল|রোগ|হৃদ|ওজন|ব্যায়াম|সুস্থ|ডায়াবেটিস|মানসিক"],"নিয়মিত|মিনিট|কদম|শরীর|পেশি|ঘণ্টা|সকাল"),
(80,"Public healthcare",["সরকারি হাসপাতাল|সরকারি স্বাস্থ্য|জনস্বাস্থ্য|স্বাস্থ্যসেবা|স্বাস্থ্য সেবা|কমিউনিটি ক্লিনিক|স্বাস্থ্য অধিদপ্তর|স্বাস্থ্য খাত|স্বাস্থ্যখাত|স্বাস্থ্যব্যবস্থা|স্বাস্থ্য ব্যবস্থা|স্বাস্থ্য কমপ্লেক্স|সরকারি চিকিৎসা|স্বাস্থ্য মন্ত্রণালয়"],"চিকিৎসক|চিকিৎসা|হাসপাতাল|রোগী|সেবা|বরাদ্দ|ওষুধ|নার্স|শয্যা|সংকট|বাজেট|ক্লিনিক|গ্রাম"),
(81,"Freelancing",["ফ্রিল্যান্সিং|ফ্রিল্যান্সার|ফ্রিল্যান্স|আপওয়ার্ক|ফাইভার"],"আয়|ডলার|আউটসোর্সিং|অনলাইন|দক্ষতা|প্রশিক্ষণ|তরুণ|রেমিট্যান্স|ক্লায়েন্ট|বিদেশি|আইসিটি|প্ল্যাটফর্ম|গ্রাফিক"),
(82,"Remote work",["রিমোট ওয়ার্ক|রিমোট জব|রিমোটলি|বাসা থেকে কাজ|বাড়ি থেকে কাজ|ঘরে বসে কাজ|ওয়ার্ক ফ্রম হোম|দূর থেকে কাজ|দূরবর্তী কাজ|হাইব্রিড কাজ|হাইব্রিড অফিস|ডব্লিউএফএইচ|বাড়িতে বসে কাজ|হোম অফিস"],"অফিস|কর্মী|কর্মচারী|করোনা|জুম|প্রযুক্তি|কর্মঘণ্টা|কোম্পানি|ইন্টারনেট|সুবিধা|উৎপাদনশীলতা|কর্মক্ষেত্র"),
(83,"Small businesses",["ক্ষুদ্র ব্যবসা|ছোট ব্যবসা|ক্ষুদ্র ও মাঝারি|এসএমই|ক্ষুদ্র উদ্যোক্তা|ক্ষুদ্রশিল্প|কুটির শিল্প|ক্ষুদ্র ঋণ|ক্ষুদ্র ব্যবসায়ী|ছোট ব্যবসায়ী|ক্ষুদ্র শিল্প|ছোট উদ্যোক্তা|ক্ষুদ্র উদ্যোগ|ছোট দোকান"],"উদ্যোক্তা|ঋণ|পুঁজি|ব্যাংক|ব্যবসায়ী|বিনিয়োগ|দোকান|বাজার|কর্মসংস্থান|প্রণোদনা|বিক্রি|লাভ|ক্ষতি"),
(84,"Youth entrepreneurship",["উদ্যোক্তা|স্টার্টআপ|স্টার্ট আপ|উদ্যোক্তা উন্নয়ন",YTH],"ব্যবসা|পুঁজি|ঋণ|বিনিয়োগ|প্রশিক্ষণ|নতুন ধারণা|উদ্ভাবন|ইনকিউবেটর|কর্মসংস্থান|সফল|প্রণোদনা"),
(85,"E-commerce businesses",["ই কমার্স|ইকমার্স|এফ কমার্স|ডিজিটাল কমার্স|অনলাইন শপ|অনলাইন ব্যবসা|অনলাইন ব্যবসায়ী|অনলাইন প্ল্যাটফর্ম|দারাজ|ই ক্যাব|পিকাবু|চালডাল|অনলাইনে বিক্রি|অনলাইন উদ্যোক্তা|অনলাইন স্টোর|মার্কেটপ্লেস","ব্যবসা|ব্যবসায়ী|উদ্যোক্তা|বিক্রেতা|প্ল্যাটফর্ম|কোম্পানি|বিনিয়োগ|মুনাফা|লোকসান|বিক্রয়|দোকান"],"অর্ডার|ডেলিভারি|গ্রাহক|লেনদেন|ফেসবুক|পণ্য|বিক্রি"),
(86,"Rising living costs",["জীবনযাত্রার ব্যয়|জীবনযাত্রার খরচ|জীবনযাপনের খরচ|ব্যয়বৃদ্ধি|খরচ বেড়|খরচ বাড়|ব্যয় বেড়|ব্যয় বাড়|জীবনধারণের ব্যয়|জীবনধারণের খরচ|বাসা ভাড়া|সংসার খরচ|সংসার চালাতে|ব্যয়ের চাপ|খরচের চাপ|জীবনের ব্যয়"],"আয়|মজুরি|বেতন|নিম্নআয়|মধ্যবিত্ত|সংসার|খরচ|ব্যয়|বাজার|ভাড়া|চাপ|সঞ্চয়|ঋণ|দ্রব্যমূল্য|নিত্যপণ্য|পরিবার"),
(87,"Inflation",["মূল্যস্ফীতি|মুদ্রাস্ফীতি|ইনফ্লেশন"],"বিবিএস|পয়েন্ট|শতাংশ|সুদের হার|বাংলাদেশ ব্যাংক|খাদ্যমূল্য|দাম|বাজার|সুদহার|ক্রয়ক্ষমতা|ডলার|অর্থনীতি|আয়|রিজার্ভ|পণ্য"),
(88,"Future employment",["চাকরি|কর্মসংস্থান|পেশা|শ্রমবাজার|কর্মবাজার|কর্মক্ষেত্র","ভবিষ্য|আগামী|চতুর্থ শিল্পবিপ্লব|2030$|2050$|নতুন দক্ষতা|উদীয়মান|প্রয়োজনীয় দক্ষতা"],"দক্ষতা|প্রযুক্তি|অটোমেশন|কৃত্রিম বুদ্ধিমত্তা|তরুণ|শিক্ষা|প্রশিক্ষণ|চাহিদা|শিল্পবিপ্লব|ফ্রিল্যান্স"),
(89,"Gig economy",["গিগ$|গিগ ইকোনমি|গিগ অর্থনীতি|গিগ ওয়ার্কার|পাঠাও|উবার|ফুডপান্ডা|ডেলিভারি কর্মী|ডেলিভারি রাইডার|ডেলিভারিম্যান|রাইড শেয়ারিং|রাইডশেয়ারিং|প্ল্যাটফর্মভিত্তিক|অ্যাপভিত্তিক|খণ্ডকালীন|অস্থায়ী কাজ"],"রাইডার|আয়|চালক|অ্যাপ|প্ল্যাটফর্ম|শ্রমিক|ডেলিভারি|কমিশন|সুরক্ষা|নিরাপত্তা|বাইক|ঘণ্টা"),
(90,"Automation and jobs",["অটোমেশন|স্বয়ংক্রিয়করণ|স্বয়ংক্রিয়|রোবট|যান্ত্রিকীকরণ|রোবটিক্স|রোবোটিক","চাকরি|কর্মসংস্থান|শ্রমিক|কর্মী|শ্রমবাজার|কর্মক্ষেত্র|বেকার|কর্মচারী|ছাঁটাই"],"প্রযুক্তি|কারখানা|পোশাক|গার্মেন্টস|দক্ষতা|উৎপাদন|মেশিন|যন্ত্র|ঝুঁকি|প্রতিস্থাপন|কৃত্রিম বুদ্ধিমত্তা"),
(91,"Traditional festivals",["উৎসব|পার্বণ|পহেলা বৈশাখ|বৈশাখী|নবান্ন|বর্ষবরণ|হালখাতা|নৌকাবাইচ|সংক্রান্তি|বিজু|সাংগ্রাই|প্রবারণা|মেলা$|মেলায়$","ঐতিহ্যবাহী|ঐতিহ্য|প্রথা|লোক|বাঙালি|চিরাচরিত|পরম্পরা|রীতি|রীতিনীতি|সংস্কৃতি"],"আদি|পুরোনো|হাজার বছরের|আয়োজন|পিঠা|শোভাযাত্রা|গ্রাম"),
(92,"Cultural heritage",["সাংস্কৃতিক ঐতিহ্য|সাংস্কৃতিক সম্পদ|হেরিটেজ|ইউনেস্কো|ঐতিহ্য সংরক্ষণ|ঐতিহ্যবাহী|বিশ্ব ঐতিহ্য|ঐতিহ্য|লোকসংস্কৃতি|জামদানি|মসলিন|পুরাকীর্তি"],"সংরক্ষণ|সংস্কৃতি|ইতিহাস|প্রাচীন|শিল্প|লোক|ঐতিহাসিক|প্রত্ন|জাদুঘর|কারুশিল্প|বাঙালি"),
(93,"Tourism in Bangladesh",["পর্যটন|পর্যটক|ট্যুরিজম|ট্যুরিস্ট|পর্যটনশিল্প|দর্শনার্থী","বাংলাদেশ|সুন্দরবন|কক্সবাজার|সিলেট|সেন্টমার্টিন|কুয়াকাটা|বান্দরবান|রাঙামাটি|খাগড়াছড়ি|সাজেক|জাফলং|পর্যটনকেন্দ্র|দর্শনীয় স্থান"],"হোটেল|রিসোর্ট|পর্যটন করপোরেশন|বিদেশি|ছুটি|আকর্ষণ|সৈকত|প্রকৃতি|বিনিয়োগ|সুবিধা"),
(94,"Cox's Bazar tourism",["কক্সবাজার|কক্স বাজার","পর্যটন|পর্যটক|পর্যটনশিল্প|সৈকত|সমুদ্রসৈকত|হোটেল|ট্যুরিস্ট|ভ্রমণ"],"লাবণী|ইনানী|হিমছড়ি|সুগন্ধা|হোটেল মোটেল|রিসোর্ট|ট্যুর|বিচ|সমুদ্র|ছুটি|মৌসুম"),
(95,"Historical sites in Bangladesh",["ঐতিহাসিক|প্রত্নতাত্ত্বিক|প্রত্নস্থান|প্রত্নতত্ত্ব|পুরাকীর্তি|ধ্বংসাবশেষ|নিদর্শন|প্রাচীন|দুর্গ|জমিদারবাড়ি|কুঠিবাড়ি|স্মৃতিসৌধ","পাহাড়পুর|মহাস্থান|ষাটগম্বুজ|লালবাগ|আহসান মঞ্জিল|কান্তজীর|সোনারগাঁও|ময়নামতি|শালবন|পুঠিয়া|মসজিদ|মন্দির|বিহার|কেল্লা|দুর্গ|জাদুঘর|রাজবাড়ি|জমিদারবাড়ি|কুঠিবাড়ি|স্থাপনা|স্মৃতিসৌধ"],"সংরক্ষণ|পর্যটক|ইতিহাস|প্রত্ন|প্রাচীন|শতাব্দী|নবাব|সুলতান"),
(96,"Social media and culture",["সামাজিক যোগাযোগমাধ্যম|সামাজিক যোগাযোগ মাধ্যম|সামাজিক মাধ্যম|সোশ্যাল মিডিয়া|ফেসবুক|ইউটিউব|টিকটক|ইনস্টাগ্রাম","সংস্কৃতি|সাংস্কৃতিক|মূল্যবোধ|ঐতিহ্য|জীবনধারা|রীতিনীতি|সংগীত|নাটক"],"তরুণ|প্রজন্ম|ট্রেন্ড|ভাইরাল|কনটেন্ট|প্রভাব|ভাষা|বিনোদন"),
(97,"Youth lifestyle changes",["জীবনযাত্রা|জীবনধারা|জীবনযাপন|লাইফস্টাইল|জীবনশৈলী|জীবনাচরণ|অভ্যাস|চালচলন|ফ্যাশন|খাদ্যাভ্যাস",YTH],"পরিবর্তন|বদলে|বদলাচ্ছে|প্রযুক্তি|সামাজিক যোগাযোগ|শহর|ডিজিটাল|আড্ডা|ফাস্ট ফুড"),
(98,"Public libraries",["পাঠাগার|লাইব্রেরি|লাইব্রেরী|গ্রন্থাগার|গণগ্রন্থাগার"],"বই|পাঠক|পড়া|জেলা|বইমেলা|সরকারি|গ্রন্থাগার অধিদপ্তর|পাঠ|শিক্ষার্থী|সংগ্রহ|ডিজিটাল"),
(99,"Sports and national identity",["খেলা|ক্রীড়া|ক্রিকেট|ফুটবল|খেলোয়াড়|ক্রিকেটার|জাতীয় দল|বিপিএল|বিশ্বকাপ|অলিম্পিক","জাতীয় পরিচয়|জাতীয়তাবাদ|দেশপ্রেম|জাতীয় গর্ব|জাতীয় ঐক্য|জাতীয় চেতনা|বাঙালিত্ব|জাতীয় সংহতি|পতাকা|জাতীয় সংগীত|দেশের গর্ব|ঐক্য|গর্ব"],"জয়|দর্শক|সমর্থক|দেশ|জার্সি|মাঠ|টুর্নামেন্ট"),
(100,"Role of journalism",["সাংবাদিকতা|সাংবাদিক|গণমাধ্যম|সংবাদমাধ্যম|সংবাদ মাধ্যম|সম্পাদক|প্রেস ক্লাব|সংবাদপত্র","ভূমিকা|দায়িত্ব|স্বাধীনতা|গণতন্ত্র|নৈতিকতা|জবাবদিহি|সত্য|নিরপেক্ষ|পেশাদারি|স্বাধীন|সমাজ|মত প্রকাশ"],"সংবাদ|প্রতিবেদন|মিডিয়া|প্রেস|সাংবাদিকদের|নিরপেক্ষতা|সাংবাদিক সংগঠন"),
]
assert len(TOPICS) == 100 and [t[0] for t in TOPICS] == list(range(1, 101))




# ================================================================
# BASIC VALIDATION
# ================================================================

assert len(TOPICS) == 100
assert [t[0] for t in TOPICS] == list(range(1, 101))

print(f"Loaded {len(TOPICS)} topics.")


# ================================================================
# TEXT NORMALIZATION
# ================================================================

_ZW = {
    0x200c: None,
    0x200d: None,
    0xfeff: None
}

_DIG = {
    ord(a): b
    for a, b in zip(
        "০১২৩৪৫৬৭৮৯",
        "0123456789"
    )
}

_TR = {
    **_ZW,
    **_DIG
}


def norm(s):
    """
    Normalize Bangla text while keeping meaning.
    """
    if not isinstance(s, str):
        return ""

    s = unicodedata.normalize(
        "NFC",
        s
    ).translate(_TR).lower()

    # Fix common Bangla Unicode ordering.
    s = re.sub(
        "\u0981([\u09be-\u09cc])",
        lambda m: m.group(1) + "\u0981",
        s
    )

    return s


TOK_RE = re.compile(
    r"[\u0980-\u09ff]+|[a-z0-9]+"
)


def bn_ratio(s):
    """
    Percentage of characters belonging to Bangla Unicode block.
    """
    s = re.sub(r"\s+", "", s)

    if not s:
        return 0.0

    return sum(
        1
        for ch in s
        if "\u0980" <= ch <= "\u09ff"
    ) / len(s)


# ================================================================
# TOPIC TERM INDEX
# ================================================================

def parse_term(term):
    """
    Convert:
        'ঢাকা'
        'ঢাকা$'
        'মেশিন লার্নিং'

    into searchable components.
    """

    parts = []

    for p in norm(term).replace("-", " ").split():

        exact = p.endswith("$")

        p = p.rstrip("$")

        if p:
            parts.append(
                (p, exact)
            )

    return tuple(parts)


TERMS = []
INDEX = defaultdict(list)
TOPIC_META = {}


for tid, english_topic, groups, support_terms in TOPICS:

    TOPIC_META[tid] = {
        "english": english_topic,
        "groups": len(groups)
    }

    all_groups = list(groups) + [support_terms]

    for gi, group in enumerate(all_groups):

        # -1 = support terms
        slot = gi if gi < len(groups) else -1

        for term in group.split("|"):

            parts = parse_term(term)

            if not parts:
                continue

            term_text = " ".join(
                p for p, _ in parts
            )

            idx = len(TERMS)

            TERMS.append(
                (
                    tid,
                    slot,
                    term_text
                )
            )

            INDEX[
                parts[0][0]
            ].append(
                (
                    parts,
                    parts[0][1],
                    tid,
                    slot,
                    idx
                )
            )


KEYLENS = sorted(
    {
        len(key)
        for key in INDEX
    }
)


def find_hits(tokens):
    """
    Find topic concept matches in tokenized text.
    """

    hits = []

    n = len(tokens)

    for i, tok in enumerate(tokens):

        L = len(tok)

        for k in KEYLENS:

            if k > L:
                break

            entries = INDEX.get(
                tok[:k]
            )

            if not entries:
                continue

            for (
                parts,
                first_exact,
                tid,
                slot,
                idx
            ) in entries:

                if first_exact and k != L:
                    continue

                ok = True

                for p in range(
                    1,
                    len(parts)
                ):

                    if i + p >= n:
                        ok = False
                        break

                    stem, exact = parts[p]

                    token2 = tokens[i + p]

                    if exact:

                        if token2 != stem:
                            ok = False
                            break

                    else:

                        if not token2.startswith(stem):
                            ok = False
                            break

                if ok:
                    hits.append(
                        (
                            tid,
                            slot,
                            idx
                        )
                    )

    return hits


def aggregate(hits):

    result = {}

    for tid, slot, idx in hits:

        if tid not in result:

            result[tid] = {
                "core": Counter(),
                "supp": Counter(),
                "groups": Counter()
            }

        if slot >= 0:

            result[tid]["core"][idx] += 1
            result[tid]["groups"][slot] += 1

        else:

            result[tid]["supp"][idx] += 1

    return result


def sentence_stats(hits):

    result = {}

    for tid, slot, idx in hits:

        if tid not in result:

            result[tid] = [
                0,  # core
                0,  # support
                0   # group bitmask
            ]

        if slot >= 0:

            result[tid][0] += 1
            result[tid][2] |= (
                1 << slot
            )

        else:

            result[tid][1] += 1

    return result


# ================================================================
# SENTENCE PROCESSING
# ================================================================

BAD_PAT = re.compile(
    r"""
    https?:
    |www\.
    |\.com\b
    |\.net\b
    |@\w
    |ছবি\s*:
    |ফাইল\s*ছবি
    |সূত্র\s*:
    |আরও\s*পড়ুন
    |বিজ্ঞাপন
    |কপিরাইট
    |অনুসরণ\s*করুন
    |ফেসবুক\s*পেজে
    |শেয়ার\s*করুন
    |মন্তব্য\s*করুন
    |বিস্তারিত\s*আসছে
    |ক্লিক\s*করুন
    |সাবস্ক্রাইব
    |আপডেট\s*:
    |প্রকাশ\s*:
    |প্রকাশিত\s*:
    |শেষ\s*আপডেট
    """,
    re.VERBOSE
)

DATELINE = re.compile(
    r"""
    ^[^।\n]{0,60}?
    \(
    [^)]{0,40}
    (?:
        প্রতিনিধি
        |প্রতিবেদক
        |ডেস্ক
        |রিপোর্টার
        |সংবাদদাতা
    )
    [^)]{0,20}
    \)
    \s*
    [:ঃ–—-]*
    \s*
    """,
    re.VERBOSE
)


def set_terminators(use_dot=False):

    global SPLIT_RE
    global TERM_END

    chars = "।?!"

    if use_dot:
        chars += "."

    SPLIT_RE = re.compile(
        "(?<=["
        + re.escape(chars)
        + "])\\s+"
    )

    TERM_END = re.compile(
        "["
        + re.escape(chars)
        + r"]['\u201d\u2019\u00bb)\]\"]*$"
    )


set_terminators(False)


def split_sentences(text):

    sentences = []

    for line in re.split(
        r"[\r\n]+",
        text
    ):

        line = line.strip()

        if not line:
            continue

        pieces = SPLIT_RE.split(
            line
        )

        for s in pieces:

            s = DATELINE.sub(
                "",
                re.sub(
                    r"\s+",
                    " ",
                    s
                )
            ).strip()

            if s:
                sentences.append(s)

    return sentences


def sentence_ok(
    sentence,
    word_count
):

    if not (
        4 <= word_count <= 100
    ):
        return False

    if not TERM_END.search(sentence):
        return False

    if BAD_PAT.search(sentence):
        return False

    if bn_ratio(sentence) < 0.75:
        return False

    return True


# ================================================================
# PASSAGE SELECTION
# ================================================================

def best_window(
    word_counts,
    valid,
    stats,
    tid,
    group_count
):

    best = None

    n = len(word_counts)

    for i in range(n):

        if not valid[i]:
            continue

        total_words = 0
        core = 0
        support = 0
        mask = 0

        for j in range(i, n):

            if not valid[j]:
                break

            total_words += word_counts[j]

            if total_words > MAX_W:
                break

            st = stats[j].get(tid)

            if st:

                core += st[0]
                support += st[1]
                mask |= st[2]

            if total_words < MIN_W:
                continue

            if core <= 0:
                continue

            coverage = (
                bin(mask).count("1")
                / group_count
            )

            length_score = (
                1 -
                min(
                    abs(
                        total_words
                        - TARGET_W
                    ) / 50,
                    1
                )
            )

            passage_score = (
                100
                * (
                    0.50 * coverage
                    + 0.35
                    * min(
                        core / 3,
                        1
                    )
                    + 0.15
                    * length_score
                )
            )

            final_score = (
                passage_score
                + 0.5
                * min(
                    support,
                    4
                )
            )

            candidate = (
                final_score,
                i,
                j,
                passage_score,
                total_words,
                coverage
            )

            if (
                best is None
                or candidate[0] > best[0]
            ):
                best = candidate

    return best


# ================================================================
# ARTICLE SCORING
# ================================================================

def article_length_score(n):

    if n < 150:

        return (
            10
            * max(
                0,
                n - MIN_ARTICLE_WORDS
            )
            / (
                150
                - MIN_ARTICLE_WORDS
            )
        )

    if n <= 1500:
        return 10

    return max(
        0,
        10
        - (
            n - 1500
        ) / 300
    )


LOW_GENRE = [
    "বিনোদন",
    "entertainment",
    "রাশিফল",
    "horoscope",
    "ক্রীড়া",
    "খেলা",
    "sports",
    "sport",
    "ছবি",
    "photo",
    "ভিডিও",
    "video"
]

# Topics where sports/entertainment-style articles
# can actually be legitimate.
GENRE_OK = {
    91,
    92,
    96,
    99
}


def preliminary_score(
    article_stats,
    group_count,
    article_words,
    title_stats
):

    core_total = sum(
        article_stats["core"].values()
    )

    distinct_terms = len(
        article_stats["core"]
    )

    score = 0

    # Distinct core concepts.
    score += (
        25
        * min(
            distinct_terms / 2,
            1
        )
    )

    # Repeated core concepts.
    score += (
        15
        * min(
            core_total / 8,
            1
        )
    )

    # Supporting terms.
    score += (
        10
        * min(
            len(
                article_stats["supp"]
            ) / 4,
            1
        )
    )

    # Concept density.
    score += (
        10
        * min(
            (
                100
                * core_total
                / max(article_words, 1)
            ) / 2,
            1
        )
    )

    # Reasonable article length.
    score += article_length_score(
        article_words
    )

    # Title match.
    if title_stats:

        if title_stats["core"]:

            score += (
                20
                * len(
                    title_stats["groups"]
                )
                / group_count
            )

        elif title_stats["supp"]:

            score += 4

    return score


# ================================================================
# CORPUS SCHEMA DETECTION
# ================================================================

TEXT_NAMES = [
    "text",
    "content",
    "article",
    "body",
    "news",
    "description",
    "news_text",
    "full_text",
    "details"
]

TITLE_NAMES = [
    "title",
    "headline",
    "heading",
    "news_title"
]

CATEGORY_NAMES = [
    "category",
    "label",
    "topic",
    "tag",
    "tags",
    "section",
    "class",
    "categories",
    "type"
]

DATE_NAMES = [
    "date",
    "published",
    "publish_date",
    "published_date",
    "datetime",
    "time",
    "timestamp",
    "date_published",
    "published_at"
]

URL_NAMES = [
    "url",
    "link",
    "source_url",
    "news_url",
    "href"
]


def open_corpus(spec):

    last_error = None

    for revision in spec["revisions"]:

        attempts = []

        if revision is None:

            attempts = [
                {},
                {
                    "trust_remote_code": True
                }
            ]

        else:

            attempts = [
                {
                    "revision": revision
                }
            ]

        for kwargs in attempts:

            try:

                return load_dataset(
                    spec["name"],
                    spec["config"],
                    streaming=True,
                    **kwargs
                )

            except Exception as e:

                last_error = e

    raise last_error


def iter_rows(dataset):

    if hasattr(dataset, "keys"):

        for split in dataset.keys():

            for row in dataset[split]:

                yield row

    else:

        yield from dataset


def get_label_maps(dataset):

    try:

        if hasattr(dataset, "keys"):

            split = dataset[
                list(dataset.keys())[0]
            ]

        else:

            split = dataset

        features = (
            getattr(
                split,
                "features",
                None
            )
            or {}
        )

        return {
            column: feature.names
            for column, feature in features.items()
            if hasattr(feature, "names")
        }

    except Exception:

        return {}


def detect_schema(rows):

    if not rows:
        raise ValueError(
            "Dataset returned no rows."
        )

    columns = list(
        rows[0].keys()
    )

    string_columns = []

    for col in columns:

        values = [
            r.get(col)
            for r in rows
        ]

        ratio = (
            sum(
                isinstance(v, str)
                for v in values
            )
            / len(values)
        )

        if ratio >= 0.8:
            string_columns.append(col)

    if not string_columns:
        raise ValueError(
            "No string columns found."
        )

    avg_length = {}

    bangla_ratio = {}

    for col in string_columns:

        vals = [
            r[col]
            for r in rows
            if isinstance(
                r.get(col),
                str
            )
        ]

        if not vals:
            continue

        avg_length[col] = statistics.mean(
            len(v)
            for v in vals
        )

        bangla_ratio[col] = statistics.mean(
            bn_ratio(v[:500])
            for v in vals
            if v
        )

    article_columns = [
        col
        for col in string_columns
        if bangla_ratio.get(col, 0) > 0.6
        and avg_length.get(col, 0) > 400
    ]

    if not article_columns:

        raise ValueError(
            "No long Bangla article column found. "
            f"Columns: {columns}"
        )

    def choose(
        possible_names,
        pool
    ):

        for wanted in possible_names:

            for col in pool:

                if col.lower() == wanted:
                    return col

        return None

    text_column = (
        choose(
            TEXT_NAMES,
            article_columns
        )
        or max(
            article_columns,
            key=lambda c:
                avg_length[c]
        )
    )

    title_column = choose(
        TITLE_NAMES,
        [
            c
            for c in columns
            if c != text_column
        ]
    )

    category_column = choose(
        CATEGORY_NAMES,
        columns
    )

    date_column = choose(
        DATE_NAMES,
        columns
    )

    url_column = choose(
        URL_NAMES,
        columns
    )

    return {
        "text": text_column,
        "title": title_column,
        "category": category_column,
        "date": date_column,
        "url": url_column
    }


# ================================================================
# CHECKPOINT STATE
# ================================================================

HEAPS = {
    tid: []
    for tid, *_ in TOPICS
}

STATE = {
    "progress": {},
    "done": []
}


if os.path.exists(CKPT):

    try:

        with open(
            CKPT,
            "rb"
        ) as f:

            checkpoint = pickle.load(f)

        HEAPS = checkpoint[
            "heaps"
        ]

        STATE = checkpoint[
            "state"
        ]

        print(
            "Checkpoint loaded."
        )

        print(
            "Progress:",
            STATE["progress"]
        )

        print(
            "Completed:",
            STATE["done"]
        )

    except Exception as e:

        print(
            "Checkpoint could not be loaded."
        )

        print(
            "Starting fresh:",
            e
        )


def save_checkpoint():

    temp_file = (
        CKPT
        + ".tmp"
    )

    with open(
        temp_file,
        "wb"
    ) as f:

        pickle.dump(
            {
                "heaps": HEAPS,
                "state": STATE
            },
            f
        )

    os.replace(
        temp_file,
        CKPT
    )


# ================================================================
# COVERAGE
# ================================================================

def coverage():

    return sum(
        1
        for heap in HEAPS.values()
        if heap
        and heap[0]["final"]
        >= ACCEPT
    )


# ================================================================
# PROCESS ONE ARTICLE
# ================================================================

def process_document(
    row,
    columns,
    label_maps,
    source
):

    text = row.get(
        columns["text"]
    )

    if not isinstance(
        text,
        str
    ):
        return

    if len(text) < 600:
        return

    if bn_ratio(
        text[:800]
    ) < 0.60:
        return

    text = text[
        :40000
    ].strip()

    # ------------------------------------------
    # TITLE
    # ------------------------------------------

    title = ""

    if columns["title"]:

        value = row.get(
            columns["title"]
        )

        if isinstance(
            value,
            str
        ):
            title = value.strip()

    # Guess title from first line if necessary.
    if not title:

        first, sep, rest = (
            text.partition("\n")
        )

        if (
            sep
            and len(first.split()) <= 25
            and not TERM_END.search(
                first.strip()
            )
        ):

            title = first.strip()
            text = rest

    # ------------------------------------------
    # ARTICLE LENGTH
    # ------------------------------------------

    article_words = len(
        text.split()
    )

    if not (
        MIN_ARTICLE_WORDS
        <= article_words
        <= MAX_ARTICLE_WORDS
    ):
        return

    # ------------------------------------------
    # FIND TOPIC CONCEPTS
    # ------------------------------------------

    tokens = TOK_RE.findall(
        norm(text)
    )

    article_aggregate = aggregate(
        find_hits(tokens)
    )

    if not article_aggregate:
        return

    title_aggregate = {}

    if title:

        title_aggregate = aggregate(
            find_hits(
                TOK_RE.findall(
                    norm(title)
                )
            )
        )

    # ------------------------------------------
    # ONLY PROCESS PLAUSIBLE TOPICS
    # ------------------------------------------

    possible_topics = []

    for tid, stats in article_aggregate.items():

        group_count = TOPIC_META[
            tid
        ]["groups"]

        # Every required concept group
        # should appear somewhere.
        if len(
            stats["groups"]
        ) < group_count:
            continue

        title_stats = (
            title_aggregate.get(tid)
        )

        # Anchor concept should appear.
        anchor_hits = stats[
            "groups"
        ].get(
            0,
            0
        )

        if (
            anchor_hits
            < MIN_CORE_HITS
            and not (
                title_stats
                and title_stats["core"]
            )
        ):
            continue

        pre_score = preliminary_score(
            stats,
            group_count,
            article_words,
            title_stats
        )

        possible_topics.append(
            (
                tid,
                stats,
                title_stats,
                pre_score,
                group_count
            )
        )

    if not possible_topics:
        return

    # ------------------------------------------
    # SENTENCE SPLITTING
    # ------------------------------------------

    sentences = split_sentences(
        text
    )

    if len(sentences) < 4:
        return

    word_counts = [
        len(s.split())
        for s in sentences
    ]

    valid = [
        sentence_ok(
            s,
            wc
        )
        for s, wc in zip(
            sentences,
            word_counts
        )
    ]

    sentence_topic_stats = []

    for sentence in sentences:

        sentence_tokens = (
            TOK_RE.findall(
                norm(sentence)
            )
        )

        sentence_hits = find_hits(
            sentence_tokens
        )

        sentence_topic_stats.append(
            sentence_stats(
                sentence_hits
            )
        )

    # ------------------------------------------
    # METADATA
    # ------------------------------------------

    category = ""

    if columns["category"]:

        value = row.get(
            columns["category"]
        )

        if (
            columns["category"]
            in label_maps
            and isinstance(
                value,
                int
            )
            and 0 <= value < len(
                label_maps[
                    columns["category"]
                ]
            )
        ):

            value = label_maps[
                columns["category"]
            ][value]

        if isinstance(
            value,
            (list, tuple)
        ):

            value = ", ".join(
                map(
                    str,
                    value
                )
            )

        if value is not None:
            category = str(value)

    date = ""

    if columns["date"]:

        value = row.get(
            columns["date"]
        )

        if value is not None:
            date = str(value)

    url = ""

    if columns["url"]:

        value = row.get(
            columns["url"]
        )

        if value is not None:
            url = str(value)

    # ------------------------------------------
    # UNIQUE ARTICLE ID
    # ------------------------------------------

    uid = hashlib.md5(
        norm(
            text[:500]
        ).encode("utf-8")
    ).hexdigest()[:16]

    # ------------------------------------------
    # BUILD CANDIDATES
    # ------------------------------------------

    for (
        tid,
        article_stats,
        title_stats,
        pre_score,
        group_count
    ) in possible_topics:

        heap = HEAPS[tid]

        # Avoid duplicate article.
        if any(
            c["uid"] == uid
            for c in heap
        ):
            continue

        # If heap already has strong candidates,
        # skip expensive passage search when article
        # clearly cannot beat them.
        if len(heap) >= K:

            current_cutoff = heap[-1][
                "final"
            ]

            rough_upper = (
                0.7
                * (pre_score + 10)
                + 30
            )

            if rough_upper <= current_cutoff:
                continue

        window = best_window(
            word_counts,
            valid,
            sentence_topic_stats,
            tid,
            group_count
        )

        if not window:
            continue

        (
            passage_score,
            start_idx,
            end_idx,
            raw_passage_score,
            passage_words,
            passage_coverage
        ) = window

        # --------------------------------------
        # CONCENTRATION
        # --------------------------------------

        if group_count == 1:

            matching_sentences = sum(
                1
                for stats in sentence_topic_stats
                if (
                    stats.get(tid)
                    and stats[tid][0] > 0
                )
            )

            fraction = (
                matching_sentences
                / max(
                    len(sentences),
                    1
                )
            )

            concentration = (
                10
                * min(
                    fraction / 0.30,
                    1
                )
            )

        else:

            all_groups_mask = (
                (1 << group_count)
                - 1
            )

            cooccurring = sum(
                1
                for stats in sentence_topic_stats
                if (
                    stats.get(tid)
                    and stats[tid][2]
                    == all_groups_mask
                )
            )

            concentration = (
                10
                * min(
                    cooccurring / 2,
                    1
                )
            )

        article_score = (
            pre_score
            + concentration
        )

        # --------------------------------------
        # GENRE PENALTY
        # --------------------------------------

        category_lower = (
            category.lower()
        )

        genre_penalty = 0

        if (
            tid not in GENRE_OK
            and any(
                bad in category_lower
                for bad in LOW_GENRE
            )
        ):

            genre_penalty = 8

        # --------------------------------------
        # FINAL SCORE
        # --------------------------------------

        final_score = (
            0.70 * article_score
            + 0.30 * raw_passage_score
            - genre_penalty
        )

        # --------------------------------------
        # NOTES
        # --------------------------------------

        core_terms = ", ".join(
            TERMS[idx][2]
            for idx, _
            in article_stats[
                "core"
            ].most_common(5)
        )

        title_match = (
            "yes"
            if (
                title_stats
                and title_stats["core"]
            )
            else "no"
        )

        notes = (
            f"article={article_score:.1f}; "
            f"passage={raw_passage_score:.1f}; "
            f"core={core_terms}; "
            f"title_match={title_match}; "
            f"passage_coverage={passage_coverage:.0%}"
        )

        if genre_penalty:
            notes += (
                f"; genre_penalty={genre_penalty}"
            )

        candidate = {
            "uid": uid,
            "topic_id": tid,
            "title": title,
            "passage": " ".join(
                sentences[
                    start_idx:
                    end_idx + 1
                ]
            ),
            "word_count": passage_words,
            "category": category,
            "date": date,
            "source": source,
            "url": url,
            "final": final_score,
            "notes": notes
        }

        heap.append(
            candidate
        )

        # Highest score first.
        heap.sort(
            key=lambda c: (
                -c["final"],
                c["uid"]
            )
        )

        # Keep only top K.
        del heap[K:]


# ================================================================
# MAIN STREAMING LOOP
# ================================================================

start_time = time.time()

stopped_early = False

PRIMARY = (
    f'{CORPORA[0]["name"]}|'
    f'{CORPORA[0]["config"]}'
)


for corpus_index, spec in enumerate(CORPORA):

    key = (
        f'{spec["name"]}|'
        f'{spec["config"]}'
    )

    if key in STATE["done"]:
        continue

    # Don't start another corpus if
    # we already have enough good topics.
    if (
        coverage()
        >= COVERAGE_TARGET
    ):
        break

    print(
        "\n"
        + "=" * 70
    )

    print(
        f"CORPUS {corpus_index + 1}: "
        f"{spec['name']}"
    )

    print(
        "=" * 70
    )

    # ------------------------------------------
    # OPEN DATASET
    # ------------------------------------------

    try:

        dataset = open_corpus(
            spec
        )

        probe = list(
            itertools.islice(
                iter_rows(dataset),
                30
            )
        )

        if not probe:
            raise ValueError(
                "No rows returned."
            )

        columns = detect_schema(
            probe
        )

        label_maps = get_label_maps(
            dataset
        )

    except Exception as e:

        print(
            f"[SKIP] Could not open "
            f"{spec['name']}"
        )

        print(
            type(e).__name__,
            str(e)[:500]
        )

        STATE["done"].append(
            key
        )

        save_checkpoint()

        continue

    # ------------------------------------------
    # SENTENCE TERMINATOR DETECTION
    # ------------------------------------------

    text_column = columns[
        "text"
    ]

    dot_counts = [
        row[text_column].count(".")
        for row in probe
        if isinstance(
            row.get(text_column),
            str
        )
    ]

    period_style = (
        statistics.mean(
            dot_counts
        )
        < 3
        if dot_counts
        else False
    )

    set_terminators(
        period_style
    )

    print(
        "Detected columns:",
        columns
    )

    print(
        "Period sentence endings:",
        period_style
    )

    print(
        "Existing coverage:",
        coverage(),
        "/ 100"
    )

    # ------------------------------------------
    # RESUME POSITION
    # ------------------------------------------

    seen = STATE[
        "progress"
    ].get(
        key,
        0
    )

    print(
        "Resuming from document:",
        seen
    )

    if seen:

        iterator = itertools.islice(
            iter_rows(dataset),
            seen,
            None
        )

    else:

        iterator = iter_rows(
            dataset
        )

    # ------------------------------------------
    # STREAM DATASET
    # ------------------------------------------

    for row in iterator:

        seen += 1

        try:

            process_document(
                row,
                columns,
                label_maps,
                key
            )

        except Exception as e:

            # Don't spam the notebook.
            if seen % 50_000 == 0:

                print(
                    "Document error:",
                    type(e).__name__,
                    str(e)[:200]
                )

        # --------------------------------------
        # CHECKPOINT
        # --------------------------------------

        if (
            seen
            % CKPT_EVERY
            == 0
        ):

            STATE[
                "progress"
            ][key] = seen

            save_checkpoint()

            elapsed = (
                time.time()
                - start_time
            ) / 60

            print(
                f"{seen:,} docs | "
                f"coverage={coverage()}/100 | "
                f"time={elapsed:.1f} min"
            )

        # --------------------------------------
        # DOCUMENT LIMIT
        # --------------------------------------

        if (
            seen
            >= MAX_DOCS_PER_CORPUS
        ):

            print(
                "Reached document limit."
            )

            break

        # --------------------------------------
        # TIME LIMIT
        # --------------------------------------

        elapsed = (
            time.time()
            - start_time
        ) / 60

        if elapsed >= TIME_LIMIT_MIN:

            stopped_early = True

            STATE[
                "progress"
            ][key] = seen

            save_checkpoint()

            print(
                "\nTime limit reached."
            )

            print(
                "Checkpoint saved."
            )

            print(
                "Run the SAME cell again "
                "to continue."
            )

            break

    # ------------------------------------------
    # HANDLE STOP
    # ------------------------------------------

    STATE[
        "progress"
    ][key] = seen

    if stopped_early:

        save_checkpoint()

        break

    # ------------------------------------------
    # CORPUS FINISHED
    # ------------------------------------------

    if key not in STATE["done"]:

        STATE[
            "done"
        ].append(key)

    save_checkpoint()

    print(
        f"\nFinished {spec['name']}"
    )

    print(
        f"Documents scanned: {seen:,}"
    )

    print(
        f"Coverage: {coverage()}/100"
    )

    if (
        coverage()
        >= COVERAGE_TARGET
    ):
        break


# ================================================================
# FINAL SELECTION
# ================================================================

print(
    "\n"
    + "=" * 70
)

print(
    "FINAL CANDIDATE SELECTION"
)

print(
    "=" * 70
)


# ------------------------------------------
# Build candidate pool.
# ------------------------------------------

pool = []

for tid, heap in HEAPS.items():

    for candidate in heap:

        bonus = (
            PRIMARY_BONUS
            if candidate["source"]
            == PRIMARY
            else 0
        )

        pool.append(
            (
                candidate["final"]
                + bonus,
                tid,
                candidate
            )
        )


pool.sort(
    key=lambda x: (
        -x[0],
        x[1],
        x[2]["uid"]
    )
)


# ------------------------------------------
# One article cannot represent
# multiple topics.
# ------------------------------------------

chosen = {}

used_articles = set()

for score, tid, candidate in pool:

    if tid in chosen:
        continue

    if candidate["uid"] in used_articles:
        continue

    chosen[tid] = candidate

    used_articles.add(
        candidate["uid"]
    )


# ================================================================
# CREATE OUTPUT TABLES
# ================================================================

selected_rows = []
candidate_rows = []


for (
    tid,
    english_topic,
    groups,
    support
) in TOPICS:

    selected = chosen.get(
        tid
    )

    # ------------------------------------------
    # SELECTED ARTICLE
    # ------------------------------------------

    if selected:

        if (
            selected["final"]
            >= ACCEPT
        ):

            status = "OK"

        else:

            status = (
                "LOW_CONFIDENCE "
                "(MANUAL REVIEW)"
            )

        selected_rows.append({

            "topic_id": tid,

            "topic": english_topic,

            "title": selected[
                "title"
            ],

            "selected_passage":
                selected[
                    "passage"
                ],

            "word_count":
                selected[
                    "word_count"
                ],

            "category":
                selected[
                    "category"
                ],

            "date":
                selected[
                    "date"
                ],

            "source":
                selected[
                    "source"
                ],

            "url":
                selected[
                    "url"
                ],

            "relevance_score":
                round(
                    selected[
                        "final"
                    ],
                    1
                ),

            "selection_notes":
                f"{status}; "
                + selected[
                    "notes"
                ]
        })

    else:

        selected_rows.append({

            "topic_id": tid,

            "topic": english_topic,

            "title": "",

            "selected_passage": "",

            "word_count": 0,

            "category": "",

            "date": "",

            "source": "",

            "url": "",

            "relevance_score": 0,

            "selection_notes":
                "NO_CANDIDATE"
        })

    # ------------------------------------------
    # ALTERNATIVE CANDIDATES
    # ------------------------------------------

    for rank, candidate in enumerate(
        HEAPS[tid][:N_ALTS],
        start=1
    ):

        candidate_rows.append({

            "topic_id": tid,

            "topic": english_topic,

            "rank": rank,

            "title":
                candidate[
                    "title"
                ],

            "candidate_passage":
                candidate[
                    "passage"
                ],

            "word_count":
                candidate[
                    "word_count"
                ],

            "category":
                candidate[
                    "category"
                ],

            "date":
                candidate[
                    "date"
                ],

            "source":
                candidate[
                    "source"
                ],

            "url":
                candidate[
                    "url"
                ],

            "relevance_score":
                round(
                    candidate[
                        "final"
                    ],
                    1
                ),

            "selected":
                bool(
                    selected
                    and candidate[
                        "uid"
                    ]
                    == selected[
                        "uid"
                    ]
                )
        })


df = pd.DataFrame(
    selected_rows
)

df_candidates = pd.DataFrame(
    candidate_rows
)


# ================================================================
# SAVE CSV FILES
# ================================================================

human_file = os.path.join(
    OUT_DIR,
    "human_articles_100.csv"
)

candidate_file = os.path.join(
    OUT_DIR,
    "human_candidates.csv"
)


df.to_csv(
    human_file,
    index=False,
    encoding="utf-8-sig"
)

df_candidates.to_csv(
    candidate_file,
    index=False,
    encoding="utf-8-sig"
)


# ================================================================
# SUMMARY
# ================================================================

selected = df[
    df["selected_passage"]
    != ""
]

n_ok = int(
    df[
        "selection_notes"
    ].str.startswith("OK")
    .sum()
)

n_low = int(
    df[
        "selection_notes"
    ].str.startswith(
        "LOW_CONFIDENCE"
    )
    .sum()
)

n_none = int(
    df[
        "selection_notes"
    ].str.startswith(
        "NO_CANDIDATE"
    )
    .sum()
)


if len(selected):

    score_bins = [
        0,
        40,
        50,
        60,
        70,
        80,
        90,
        101
    ]

    score_distribution = (
        pd.cut(
            selected[
                "relevance_score"
            ],
            bins=score_bins,
            right=False
        )
        .value_counts()
        .sort_index()
    )

else:

    score_distribution = (
        pd.Series(
            dtype=int
        )
    )


summary_lines = [

    "STAGE 1 SUMMARY",

    "",

    f"Confident selections "
    f"(score >= {ACCEPT}): "
    f"{n_ok}",

    f"Low-confidence selections: "
    f"{n_low}",

    f"No candidate: "
    f"{n_none}",

    "",

    f"Topics with any selected "
    f"passage: {len(selected)} / 100",

    f"Coverage >= ACCEPT: "
    f"{coverage()} / 100",

]


if len(selected):

    summary_lines.extend([

        "",

        "PASSAGE LENGTH",

        f"Mean: "
        f"{selected['word_count'].mean():.1f}",

        f"Minimum: "
        f"{selected['word_count'].min()}",

        f"Maximum: "
        f"{selected['word_count'].max()}",

        "",

        "SCORE DISTRIBUTION"
    ])

    for bucket, count in (
        score_distribution.items()
    ):

        summary_lines.append(
            f"{str(bucket):>15}: {count}"
        )


review_topics = df[
    ~df[
        "selection_notes"
    ].str.startswith("OK")
][
    "topic_id"
].tolist()


summary_lines.extend([

    "",

    "TOPICS TO MANUALLY REVIEW:",

    str(review_topics),

])


if stopped_early:

    summary_lines.extend([

        "",

        "The run stopped at the "
        "time limit.",

        "Run the cell again to "
        "continue from checkpoint."

    ])


summary = "\n".join(
    summary_lines
)


summary_file = os.path.join(
    OUT_DIR,
    "human_summary.txt"
)

with open(
    summary_file,
    "w",
    encoding="utf-8"
) as f:

    f.write(summary)


# ================================================================
# FINAL OUTPUT
# ================================================================

print(
    "\n" + summary
)

print(
    "\n"
    + "=" * 70
)

print(
    "FILES CREATED"
)

print(
    "=" * 70
)

print(
    human_file
)

print(
    candidate_file
)

print(
    summary_file
)

print(
    CKPT
)


print(
    "\nFirst 5 selected rows:"
)

display(
    df.head()
)


Loaded 100 topics.

CORPUS 1: thedeba/bnnews


README.md:   0%|          | 0.00/7.98k [00:00<?, ?B/s]

Detected columns: {'text': 'content', 'title': 'title', 'category': 'category', 'date': None, 'url': None}
Period sentence endings: True
Existing coverage: 0 / 100
Resuming from document: 0
25,000 docs | coverage=100/100 | time=1.9 min
50,000 docs | coverage=100/100 | time=3.6 min
75,000 docs | coverage=100/100 | time=5.1 min
100,000 docs | coverage=100/100 | time=6.7 min
125,000 docs | coverage=100/100 | time=8.2 min
150,000 docs | coverage=100/100 | time=9.7 min
175,000 docs | coverage=100/100 | time=11.3 min
200,000 docs | coverage=100/100 | time=12.9 min
225,000 docs | coverage=100/100 | time=14.4 min
250,000 docs | coverage=100/100 | time=15.9 min
275,000 docs | coverage=100/100 | time=17.6 min

Finished thedeba/bnnews
Documents scanned: 295,020
Coverage: 100/100

FINAL CANDIDATE SELECTION

STAGE 1 SUMMARY

Confident selections (score >= 50): 100
Low-confidence selections: 0
No candidate: 0

Topics with any selected passage: 100 / 100
Coverage >= ACCEPT: 100 / 100

PASSAGE LENGTH


,topic_id,topic,title,selected_passage,word_count,category,date,source,url,relevance_score,selection_notes
0,1,Traffic congestion in Dhaka,ঢাকা-চট্টগ্রাম মহাসড়কে ১২ ঘণ্টার যানজটে নাকাল ...,এ কারণে যানজটের সৃষ্টি হয়। এ ছাড়া ফৌজদারহাট বা...,150,জেলা,,thedeba/bnnews|None,,100.0,"OK; article=100.0; passage=100.0; core=যানজট, ..."
1,2,Air pollution in Bangladesh,"বায়ুদূষণে ঢাকার অবস্থান নবম, মান ‘গ্রহণযোগ্য’",বাতাসের এই মান ‘গ্রহণযোগ্য’ বা ‘মাঝারি’ মানের ...,150,পরিবেশ,,thedeba/bnnews|None,,100.0,OK; article=100.0; passage=100.0; core=বায়ুদূ...
2,3,Water pollution in Bangladesh,ভরাট-দখলে বিলীন বুড়িগঙ্গার ১৬ কিলোমিটার: গবেষণা,"গবেষণায় দেখা গেছে, কেরানীগঞ্জের হযরতপুর ইউনিয়ন...",150,রাজধানী,,thedeba/bnnews|None,,100.0,OK; article=100.0; passage=100.0; core=বুড়িগঙ...
3,4,Plastic waste management,প্লাস্টিক বর্জ্যের ক্ষতির দিক আবারও মনে করিয়ে ...,ফলে বিশ্বের অন্যান্য অনেক দেশের মতো সমুদ্রের প...,150,বিশ্ব,,thedeba/bnnews|None,,100.0,OK; article=100.0; passage=100.0; core=প্লাস্ট...
4,5,Flooding in Bangladesh,"তিস্তাসহ ৩ নদীর পানি দ্রুত বাড়তে পারে, ৪ জেলায়...","এটি এখন ভারতের পশ্চিমবঙ্গ, ওডিশা ও বিহার রাজ্য...",150,পরিবেশ,,thedeba/bnnews|None,,100.0,"OK; article=100.0; passage=100.0; core=বন্যা, ..."
